In [ ]:
import pathlib
import obspy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hvsrpy
from hvsrpy.sesame import reliability, clarity
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
from scipy.stats import skew
import joblib

plt.style.use(hvsrpy.HVSRPY_MPL_STYLE)

In [ ]:
elevation_in_m = 3
elevation_1500m_avg_in_m = -4.320

file_path = pathlib.Path(
    "D:/Document/Kuliah/Bismillah TA/Data/DATA 2 (Meulaboh)/"
    "Raw waveform microtremor data recorded in the City of Meulaboh/"
    "GL1-GEOBIT-60minutes.mseed"
)

if not file_path.exists():
    raise FileNotFoundError(f"File {file_path} not found.")

print("File exists.")

In [ ]:
stream = obspy.read(str(file_path))

if len(stream) != 3:
    raise ValueError(
        f"Recording must contain exactly 3 components (Z, N, E), "
        f"but found {len(stream)}."
    )

start_times = [tr.stats.starttime for tr in stream]
end_times   = [tr.stats.endtime for tr in stream]

common_start = max(start_times)
common_end   = min(end_times)
common_duration = common_end - common_start

if common_duration <= 0:
    raise ValueError("No overlapping time window among components.")

stream.trim(common_start, common_end)

print(f"Common duration: {common_duration / 60:.2f} minutes")

# Save trimmed file
output_path = file_path.with_name(file_path.stem + "_updated.mseed")
stream.write(str(output_path), format="MSEED")

print(f"Trimmed file saved as:\n{output_path}")

In [ ]:
preprocessing_settings = hvsrpy.settings.HvsrPreProcessingSettings()
preprocessing_settings.detrend = "constant"
preprocessing_settings.window_length_in_seconds = 20
preprocessing_settings.orient_to_degrees_from_north = 0.0
preprocessing_settings.filter_corner_frequencies_in_hz = (0.5, 10)
preprocessing_settings.ignore_dissimilar_time_step_warning = False

print("Preprocessing Summary")
print("-"*60)
preprocessing_settings.psummary()

In [ ]:
processing_settings = hvsrpy.settings.HvsrTraditionalProcessingSettings()
processing_settings.window_type_and_width = ("tukey", 0.1)
processing_settings.smoothing=dict(operator="konno_and_ohmachi",
                                   bandwidth=40,
                                   center_frequencies_in_hz=np.geomspace(0.5, 10, 128))
processing_settings.method_to_combine_horizontals = "squared_average"
processing_settings.handle_dissimilar_time_steps_by = "frequency_domain_resampling"

print("Processing Summary")
print("-"*60)
processing_settings.psummary()

In [ ]:
updated_file = output_path

srecords = hvsrpy.read([str(updated_file)])

srecords_preprocessed = hvsrpy.preprocess(srecords, preprocessing_settings)
hvsr = hvsrpy.process(srecords_preprocessed, processing_settings)

In [ ]:
# Cox et al. (2020) | Frequency-Domain Window Rejection Algorithm
n = 1.4
search_range_in_hz = (None, None)
_ = hvsrpy.frequency_domain_window_rejection(hvsr, n=n, search_range_in_hz=search_range_in_hz)

# STA-LTA | Short term average - Long term average rejection algorithm
#srecords = hvsrpy.read(file_path)
#srecords_preprocessed = hvsrpy.preprocess(srecords, preprocessing_settings)
#_ = hvsrpy.sta_lta_window_rejection(srecords_preprocessed, hvsr=hvsr)
#hvsrpy.maximum_value_window_rejection(srecords_preprocessed, hvsr=hvsr)

# Max Value | Maximum value window rejection
#srecords = hvsrpy.read(file_path)
#srecords_preprocessed = hvsrpy.preprocess(srecords, preprocessing_settings)
#_ = hvsrpy.maximum_value_window_rejection(srecords_preprocessed, hvsr=hvsr)

# Manual | Manual value window rejection
%matplotlib tk
_ = hvsrpy.manual_window_rejection(hvsr)

In [ ]:
%matplotlib inline
mfig, axs = hvsrpy.plot_pre_and_post_rejection(srecords_preprocessed, hvsr)
plt.show()

# =========================
# WINDOW STATISTICS
# =========================

# jumlah window = panjang boolean mask
total_windows = len(hvsr.valid_window_boolean_mask)

# jumlah window valid (True)
valid_windows = np.sum(hvsr.valid_window_boolean_mask)

# rejected
rejected_windows = total_windows - valid_windows

print("\nWindow Summary:")
print("-"*25)
print(f"Total window   : {total_windows}")
print(f"Valid window   : {valid_windows}")
print(f"Rejected window: {rejected_windows}")

In [ ]:
save_figure = False
save_results = False
file_path_prefix = "example_mhvsr_traditional_window_rejection"

if save_figure:
    file_path = f"{file_path_prefix}_all_panels.png"
    mfig.savefig(file_path)
    plt.close()
    print(f"Figure saved successfully to {file_path}!")

if save_results:
    file_path = f"{file_path_prefix}.csv"
    hvsrpy.object_io.write_hvsr_object_to_file(hvsr, file_path)
    print(f"Results saved successfully to {file_path}!")

In [ ]:
print("\nStatistical Summary:")
print("-"*20)
hvsrpy.summarize_hvsr_statistics(hvsr)
(sfig, ax) = hvsrpy.plot_single_panel_hvsr_curves(hvsr)
ax.get_legend().remove()
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

search_range_in_hz = (None, None)
verbose = 1
rejection_mask = hvsr.valid_window_boolean_mask.copy()
hvsr.update_peaks_bounded(search_range_in_hz=search_range_in_hz)
hvsr.valid_window_boolean_mask = rejection_mask & hvsr.valid_peak_boolean_mask
hvsr.valid_peak_boolean_mask   = rejection_mask & hvsr.valid_peak_boolean_mask

print("\nSESAME (2004) Clarity and Reliability Criteria:")
print("-"*47)
reliability(
    windowlength=preprocessing_settings.window_length_in_seconds,
    passing_window_count=int(np.sum(hvsr.valid_window_boolean_mask)),
    frequency=hvsr.frequency,
    mean_curve=hvsr.mean_curve(distribution="lognormal"),
    std_curve=hvsr.std_curve(distribution="lognormal"),
    search_range_in_hz=search_range_in_hz,
    verbose=verbose,
)
clarity(
    frequency=hvsr.frequency,
    mean_curve=hvsr.mean_curve(distribution="lognormal"),
    std_curve=hvsr.std_curve(distribution="lognormal"),
    fn_std=hvsr.std_fn_frequency(distribution="normal"),
    search_range_in_hz=search_range_in_hz,
    verbose=verbose,
)

In [ ]:
# --- EXPORT KE FORMAT .HV (GEOPSY STYLE) ---
output_hv_path = output_path.with_suffix(".hv")

# Ambil mean curve (lognormal sesuai standar HVSR)
freq = hvsr.frequency
mean_curve = hvsr.mean_curve(distribution="lognormal")

with open(output_hv_path, "w") as f:
    f.write("# HVSR curve exported from hvsrpy\n")
    f.write("# Frequency(Hz)\tAmplitude\n")
    
    for f0, amp in zip(freq, mean_curve):
        f.write(f"{f0:.6f}\t{amp:.6f}\n")

print(f"\nFile .hv berhasil dibuat:\n{output_hv_path}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# =====================================================
# PARAMETER UTAMA
# =====================================================
hv_file = r"D:/Document/Kuliah/Bismillah TA/Data/DATA 2 (Meulaboh)/Raw waveform microtremor data recorded in the City of Meulaboh/GL1-GEOBIT-60minutes_updated.hv"

V0 = 175        # Vs permukaan (m/s)
x = 0.099       # eksponen empiris Vs(z)
bandwidth = 40 # Konno–Ohmachi
vs_tol = 0.10  # toleransi Vs Dinver (±10%)
max_layer = 15  # jumlah lapisan maksimum

# =====================================================
# 1. BACA FILE HVSR (ROBUST)
# =====================================================
freq, amp = [], []

with open(hv_file, "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        line = line.replace(",", " ")
        parts = line.split()

        nums = []
        for p in parts:
            try:
                nums.append(float(p))
            except:
                pass

        if len(nums) >= 2:
            freq.append(nums[0])
            amp.append(nums[1])

freq = np.array(freq)
amp = np.array(amp)

if len(freq) == 0:
    raise ValueError("❌ Tidak ada data numerik yang terbaca dari file HV")

# =====================================================
# 2. FILTER & SORT DATA
# =====================================================
mask = freq > 0
freq = freq[mask]
amp = amp[mask]

idx = np.argsort(freq)
freq = freq[idx]
amp = amp[idx]

print(f"✅ Data HVSR terbaca: {len(freq)} titik")

# =====================================================
# 3. KONNO–OHMACHI SMOOTHING
# =====================================================
def konno_ohmachi(f, a, bandwidth=40):
    smoothed = np.zeros_like(a)
    for i in range(len(f)):
        xko = bandwidth * np.log10(f / f[i])
        w = (np.sin(xko) / xko) ** 4
        w[xko == 0] = 1
        smoothed[i] = np.sum(w * a) / np.sum(w)
    return smoothed

amp_ko = konno_ohmachi(freq, amp, bandwidth)

# =====================================================
# 4. KONTRAS & KEDALAMAN
# =====================================================
contrast = amp - amp_ko
depth = V0 / (4 * freq)

df = pd.DataFrame({
    "Frequency (Hz)": freq,
    "Depth (m)": depth,
    "Amplitude": amp,
    "KO": amp_ko,
    "Contrast": contrast
})

# =====================================================
# 5. DETEKSI BATAS LAPISAN (PEAK KONTRAS)
# =====================================================
abs_contrast = np.abs(contrast)
peaks, _ = find_peaks(abs_contrast, distance=3)

if len(peaks) == 0:
    print("⚠️ Tidak ada peak kontras → 1 lapisan")
    layer_depths = [0, depth.max()]
else:
    strongest = peaks[np.argsort(abs_contrast[peaks])[-max_layer:]]
    layer_depths = [0]
    layer_depths += list(depth[strongest])
    layer_depths.append(depth.max())

layer_depths = sorted(set(layer_depths))

# =====================================================
# 6. TABEL INITIAL MODEL LAPISAN
# =====================================================
layers = []
for i in range(len(layer_depths) - 1):
    layers.append({
        "Layer": i + 1,
        "Top (m)": layer_depths[i],
        "Bottom (m)": layer_depths[i + 1],
        "Thickness (m)": layer_depths[i + 1] - layer_depths[i]
    })

df_layers = pd.DataFrame(layers)

print("\n📐 INITIAL MODEL LAPISAN\n")
print(df_layers.to_string(index=False, float_format="%.2f"))

# =====================================================
# 7. PLOT KONTRAS vs KEDALAMAN
# =====================================================
plt.figure(figsize=(6,8))
plt.plot(contrast, depth, color="orange", label="Amplitude Contrast")
plt.axvline(0, color="gray", linestyle="--")

for d in layer_depths:
    plt.axhline(d, color="purple", linestyle="--", linewidth=1)

plt.gca().invert_yaxis()
plt.xlabel("Amplitude Contrast")
plt.ylabel("Depth (m)")
plt.title("Layer Boundary Detection from HVSR")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# =====================================================
# 8. INITIAL Vs MODEL (STEP – GEOPSY STYLE)
# =====================================================
plt.figure(figsize=(6,8))

vs_step, z_step = [], []

for i in range(len(layer_depths) - 1):
    z_top = layer_depths[i]
    z_bot = layer_depths[i + 1]
    z_mid = 0.5 * (z_top + z_bot)

    vs_mid = V0 * (1 + z_mid) ** x

    vs_step.extend([vs_mid, vs_mid])
    z_step.extend([z_top, z_bot])

    plt.text(vs_mid + 5, z_mid, f"{vs_mid:.0f} m/s", va="center", fontsize=9)

plt.plot(vs_step, z_step, drawstyle="steps-post", color="red", linewidth=2)
plt.gca().invert_yaxis()
plt.xlabel("Vs (m/s)")
plt.ylabel("Depth (m)")
plt.title("Initial Layered Vs Model")
plt.grid(True)
plt.tight_layout()
plt.savefig("GL1.png", dpi=300)
plt.show()

# =====================================================
# 9. TABEL INPUT DINVER
# =====================================================
dinver = []

for i in range(len(layer_depths) - 1):
    z_top = layer_depths[i]
    z_bot = layer_depths[i + 1]
    z_mid = 0.5 * (z_top + z_bot)

    vs_mid = V0 * (1 + z_mid) ** x

    dinver.append({
        "Layer": i + 1,
        "Depth Range (m)": f"{z_top:.1f}-{z_bot:.1f}",
        "Thickness (m)": z_bot - z_top,
        "Vs Range (m/s)": f"{vs_mid*(1-vs_tol):.1f}-{vs_mid*(1+vs_tol):.1f}",
        "Vp (m/s)": round(1.73 * vs_mid, 1),
        "Density (g/cm3)": 1.2
    })

df_dinver = pd.DataFrame(dinver)
df_dinver.to_csv("GL1.csv", index=False)

print("\n📊 TABEL INPUT DINVER\n")
print(df_dinver.to_string(index=False))
print("\n✅ Output selesai: GL1.png dan GL1.csv")

In [ ]:
import numpy as np
from scipy.signal import find_peaks

# =====================================================
# PARAMETER UTAMA
# =====================================================
hv_file   = r"D:/Document/Kuliah/Bismillah TA/Data/DATA 2 (Meulaboh)/Raw waveform microtremor data recorded in the City of Meulaboh/GL1-GEOBIT-60minutes_updated.hv"
out_param = "GL01.param"

V0 = 175        # Vs permukaan (m/s)
x = 0.099       # eksponen empiris Vs(z)
bandwidth = 40 # Konno–Ohmachi
vs_tol = 0.10  # toleransi Vs Dinver (±10%)
max_layer = 5  # jumlah lapisan maksimum
vp_ratio  = 1.73

rho_min, rho_max = 1000, 2000
nu_min,  nu_max  = 0.2, 0.5

# =====================================================
# 1. BACA & FILTER DATA HVSR
# =====================================================
freq, amp = [], []
with open(hv_file, "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.replace(",", " ").split()
        nums = []
        for p in parts:
            try:
                nums.append(float(p))
            except ValueError:
                pass
        if len(nums) >= 2:
            freq.append(nums[0])
            amp.append(nums[1])

freq = np.array(freq)
amp = np.array(amp)
mask = freq > 0
freq, amp = freq[mask], amp[mask]
idx = np.argsort(freq)
freq, amp = freq[idx], amp[idx]

# =====================================================
# 2. KONNO-OHMACHI SMOOTHING & KONTRAS
# =====================================================
def konno_ohmachi(f, a, bandwidth=40):
    smoothed = np.zeros_like(a)
    for i in range(len(f)):
        xko = bandwidth * np.log10(f / f[i])
        w = (np.sin(xko) / xko) ** 4
        w[xko == 0] = 1
        smoothed[i] = np.sum(w * a) / np.sum(w)
    return smoothed

amp_ko = konno_ohmachi(freq, amp, bandwidth)
contrast = amp - amp_ko
depth = V0 / (4 * freq)

# =====================================================
# 3. DETEKSI BATAS LAPISAN
# =====================================================
abs_contrast = np.abs(contrast)
peaks, _ = find_peaks(abs_contrast, distance=3)

if len(peaks) == 0:
    layer_depths = [0, depth.max()]
else:
    strongest = peaks[np.argsort(abs_contrast[peaks])[-max_layer:]]
    layer_depths = [0] + list(depth[strongest]) + [depth.max()]

layer_depths = sorted(set(layer_depths))
n_layers = len(layer_depths) - 1

# =====================================================
# 4. BANGUN NILAI TIAP LAYER (Vs, Vp)
# =====================================================
vs_layers, vp_layers = [], []

for i in range(n_layers):
    z_top = layer_depths[i]
    z_bot = layer_depths[i + 1]
    z_mid = 0.5 * (z_top + z_bot)
    is_half = (i == n_layers - 1)

    vs_mid = V0 * (1 + z_mid) ** x
    vs_min = vs_mid * (1 - vs_tol)
    vs_max = vs_mid * (1 + vs_tol)
    vp_val = vp_ratio * vs_mid

    dh_min = 1 if is_half else z_top
    dh_max = 1000 if is_half else z_bot

    vs_layers.append((f"Vs{i}", vs_min, vs_max, dh_min, dh_max))
    vp_layers.append((f"Vp{i}", vp_val, vp_val, dh_min, dh_max))

# =====================================================
# 5. TEMPLATE XML (struktur persis dari contoh .param asli)
# =====================================================
def layer_block(name, top_min, top_max, dh_min, dh_max, indent=6):
    sp = " " * indent
    return (
        f'{sp}<ParamLayer name="{name}">\n'
        f'{sp}  <shape>Uniform</shape>\n'
        f'{sp}  <lastParamCondition>true</lastParamCondition>\n'
        f'{sp}  <nSubayers>1</nSubayers>\n'
        f'{sp}  <topMin>{top_min}</topMin>\n'
        f'{sp}  <topMax>{top_max}</topMax>\n'
        f'{sp}  <linkedTo>Not linked</linkedTo>\n'
        f'{sp}  <isDepth>true</isDepth>\n'
        f'{sp}  <dhMin>{dh_min}</dhMin>\n'
        f'{sp}  <dhMax>{dh_max}</dhMax>\n'
        f'{sp}</ParamLayer>\n'
    )

xml_parts = []
xml_parts.append("<Dinver>\n")
xml_parts.append("  <pluginTag>DispersionCurve</pluginTag>\n")
xml_parts.append("  <pluginTitle>Surface Wave Inversion</pluginTitle>\n")
xml_parts.append("  <ParamGroundModel>\n")
xml_parts.append("    <position>0 0 0</position>\n")

# --- Rho (Density) ---
xml_parts.append("    <ParamProfile>\n")
xml_parts.append("      <type>Param</type>\n")
xml_parts.append("      <longName>Density</longName>\n")
xml_parts.append("      <shortName>Rho</shortName>\n")
xml_parts.append("      <unit>kg/m3</unit>\n")
xml_parts.append(f"      <defaultMinimum>{rho_min}</defaultMinimum>\n")
xml_parts.append(f"      <defaultMaximum>{rho_max}</defaultMaximum>\n")
xml_parts.append("      <defaultCondition>LessThan</defaultCondition>\n")
xml_parts.append(layer_block("Rho0", rho_min, rho_max, 1, 1000))
xml_parts.append("    </ParamProfile>\n")

# --- Vs (Shear-wave velocity) ---
xml_parts.append("    <ParamProfile>\n")
xml_parts.append("      <type>Param</type>\n")
xml_parts.append("      <longName>Shear-wave velocity</longName>\n")
xml_parts.append("      <shortName>Vs</shortName>\n")
xml_parts.append("      <unit>m/s</unit>\n")
xml_parts.append("      <defaultMinimum>50</defaultMinimum>\n")
xml_parts.append("      <defaultMaximum>3500</defaultMaximum>\n")
xml_parts.append("      <defaultCondition>LessThan</defaultCondition>\n")
for name, vmin, vmax, dmin, dmax in vs_layers:
    xml_parts.append(layer_block(name, vmin, vmax, dmin, dmax))
xml_parts.append("    </ParamProfile>\n")

# --- Nu (Poisson's Ratio) ---
xml_parts.append("    <ParamProfile>\n")
xml_parts.append("      <type>Condition</type>\n")
xml_parts.append("      <longName>Poisson&apos;s Ratio</longName>\n")
xml_parts.append("      <shortName>Nu</shortName>\n")
xml_parts.append("      <unit></unit>\n")
xml_parts.append(f"      <defaultMinimum>{nu_min}</defaultMinimum>\n")
xml_parts.append(f"      <defaultMaximum>{nu_max}</defaultMaximum>\n")
xml_parts.append("      <defaultCondition>GreaterThan</defaultCondition>\n")
xml_parts.append(layer_block("Nu0", nu_min, nu_max, 1, 1000))
xml_parts.append("    </ParamProfile>\n")

# --- Vp (Compression-wave velocity) ---
xml_parts.append("    <ParamProfile>\n")
xml_parts.append("      <type>Param</type>\n")
xml_parts.append("      <longName>Compression-wave velocity</longName>\n")
xml_parts.append("      <shortName>Vp</shortName>\n")
xml_parts.append("      <unit>m/s</unit>\n")
xml_parts.append("      <defaultMinimum>200</defaultMinimum>\n")
xml_parts.append("      <defaultMaximum>5000</defaultMaximum>\n")
xml_parts.append("      <defaultCondition>LessThan</defaultCondition>\n")
for name, vmin, vmax, dmin, dmax in vp_layers:
    xml_parts.append(layer_block(name, vmin, vmax, dmin, dmax))
xml_parts.append("    </ParamProfile>\n")

xml_parts.append("  </ParamGroundModel>\n")
xml_parts.append("</Dinver>\n")

xml_content = "".join(xml_parts)

# =====================================================
# 6. SIMPAN SEBAGAI .param (UTF-16LE dengan BOM, sesuai file asli)
# =====================================================
with open(out_param, "w", encoding="utf-16") as f:
    f.write(xml_content)

print(f"✅ File .param berhasil dibuat: {out_param}")
print(f"   Jumlah layer Vs/Vp: {n_layers}")